# Assignment 2B — RAG Pipeline on the BFSI Regulatory Corpus

**Group 22** · Domain: Indian BFSI regulatory texts (RBI, SEBI, IRDAI, Basel Committee, FATF) · Re-uses the Assignment 1 corpus, evaluation questions and QLoRA adapter.

**Team Detail**

| | BITS ID | Name | Contribution |
|---|---|---|---|
| 1. | 2025AE05262 | GARIMA SRIVASTAVA | 100% |
| 2. | 2025AE05297 | KUMAR GAURAV | 100% |
| 3. | 2025AE05366 | KUMAR PRANAV | 100% |
| 4. | 2025AE05756 | SREEJA V J | 100% |

**Run environment:** BITS lab infrastructure (Prayogshala JupyterLab), NVIDIA A100-SXM4-80GB, 1 November 2026.
The notebook was run top to bottom on the lab GPU; every table and answer below is that run's output.

**Project layout** (the project folder is the notebook's own folder unless the environment variable `PROJECT_DIR` points elsewhere):

```
<project folder>/
    Assignment_2B_RAG_Pipeline.ipynb
    domain_corpus/*.txt          # the 7 cleaned Assignment 1 corpus files
    eval.jsonl, train.jsonl      # Assignment 1 dataset splits
    adapter_B_r16/               # Assignment 1 QLoRA adapter: adapter_config.json, adapter_model.safetensors
```

The GPU and disk are shared with other users, so the model-loading cells check free GPU memory and free disk first and report what
they chose or skipped. The gated Llama-3.1 judge is used only when a Hugging Face token with access is available; otherwise an open
judge is chosen (Part D2).

## Block 0 — Environment and configuration

In [2]:
# Install what is missing and repair a broken stack, without touching torch (the lab image's CUDA build).
#  1. Missing packages are installed with every already-installed core package pinned (pip constraints file), so nothing
#     else is upgraded as a side effect.
#  2. transformers is held at 5.18.0 while torch is older than 2.6: later releases call torch.accelerator, which torch 2.5
#     lacks (this broke Assignment 1 on this lab on 08-Oct-2026).
#  3. A fresh interpreter imports and exercises the whole stack. A NumPy / SciPy / pandas fault is repaired by a CLEAN
#     reinstall of the matched set proven on this lab (uninstall, delete leftover files, install fresh); a forced reinstall
#     over a damaged install can leave mixed files behind.
import importlib.util, importlib.metadata as md, json, re, shutil, site, subprocess, sys, sysconfig, tempfile
from pathlib import Path

NEED = {"sentence_transformers": "sentence-transformers", "faiss": "faiss-cpu", "rank_bm25": "rank_bm25",
        "peft": "peft", "bitsandbytes": "bitsandbytes", "accelerate": "accelerate", "sklearn": "scikit-learn"}
CORE = ["torch", "transformers", "tokenizers", "huggingface-hub", "numpy", "scipy", "pandas"]
STACK = ("numpy", "pandas", "scipy")
MATCHED_SET = ["numpy==1.26.4", "pandas==2.2.3", "scipy==1.13.1"]       # proven together on this lab (Python 3.11)
TF_PIN = "transformers==5.18.0"
USER_SITE = site.getusersitepackages()

def installed(dist):
    try:
        return md.version(dist)
    except md.PackageNotFoundError:
        return None

vt = lambda v: tuple(int(x) for x in re.findall(r"\d+", v or "0")[:3])
OLD_TORCH = vt(installed("torch")) < (2, 6)

def pip_install(*args, unpin=()):
    # pip install with every installed core package pinned, except those in `unpin`
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as fh:
        fh.write("\n".join(f"{d}=={installed(d)}" for d in CORE if installed(d) and d not in unpin))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", fh.name, *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"pip install {' '.join(args)} failed:\n{(r.stdout + r.stderr)[-2000:]}")

def probe():
    # fresh interpreter: modules already loaded in this kernel cannot hide a broken install
    code = r'''
import json, traceback
try:
    import numpy, pandas, scipy.special, scipy.stats, torch, sklearn.metrics
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    import sentence_transformers, faiss, rank_bm25, peft, bitsandbytes, accelerate
    pandas.DataFrame({"x": numpy.arange(3)}).describe(); scipy.stats.norm.cdf(0.0)
    print(json.dumps({"ok": True}))
except BaseException:
    seen = []
    for line in traceback.format_exc().splitlines():
        if line not in seen:
            seen.append(line)
    print(json.dumps({"ok": False, "error": "\n".join(seen[-30:])}))
'''
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    try:
        return json.loads(r.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return {"ok": False, "error": (r.stderr or r.stdout).strip()[-2500:]}

def clean_reinstall_stack():
    for pkg in STACK:                                                    # 1. every recorded copy
        for _ in range(3):
            r = subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", pkg], capture_output=True, text=True)
            if "not installed" in (r.stdout + r.stderr).lower():
                break
    roots = {Path(sysconfig.get_paths()["purelib"]), Path(sysconfig.get_paths()["platlib"]), Path(USER_SITE)}
    for root in roots:                                                   # 2. files no install record covers
        if root.is_dir():
            for p in root.iterdir():
                stem = p.name.split("-")[0].lower()
                if stem in STACK or stem in {f"{m}.libs" for m in STACK}:
                    print(f"  removing leftover {p}")
                    shutil.rmtree(p) if p.is_dir() else p.unlink()
    pip_install("--no-cache-dir", *MATCHED_SET, unpin=STACK)            # 3. the matched set, fresh

print("Core stack before:", {d: installed(d) for d in CORE})
changed = False
missing = [pip for mod, pip in NEED.items() if importlib.util.find_spec(mod) is None]
tf = installed("transformers")
if tf is None or (OLD_TORCH and vt(tf) != vt(TF_PIN)):
    missing.insert(0, TF_PIN if OLD_TORCH else "transformers")
if missing:
    print("Installing:", missing)
    pip_install(*missing, unpin=("transformers", "tokenizers", "huggingface-hub") if "transformers" in missing[0] else ())
    changed = True

for attempt in range(3):
    info = probe()
    if info["ok"]:
        break
    err = info["error"]
    print(f"Stack check failed:\n  " + err.replace("\n", "\n  "))
    if any(f"/{m}/" in err for m in STACK) or "numpy.ufunc" in err or "numpy.dtype size changed" in err:
        print(f"Clean reinstall of {MATCHED_SET}:")
        clean_reinstall_stack(); changed = True
    elif OLD_TORCH and "/transformers/" in err and vt(installed("transformers")) != vt(TF_PIN):
        print(f"Installing {TF_PIN}")
        pip_install(TF_PIN, unpin=("transformers", "tokenizers", "huggingface-hub")); changed = True
    else:
        raise RuntimeError("The stack fails for a reason this cell does not repair - see the traceback above.")
else:
    info = probe()                                                       # after the last repair
    if not info["ok"]:
        raise RuntimeError("Still failing after repair:\n" + info["error"])

print("Stack check passed:", {d: installed(d) for d in CORE})
if changed and any(m in sys.modules for m in ("numpy", "torch", "transformers", "scipy", "pandas")):
    raise SystemExit("Packages changed while this kernel had them loaded - restart the kernel and run the notebook from the top.")

Core stack before: {'torch': '2.5.1+cu124', 'transformers': '5.22.1', 'tokenizers': '0.23.3', 'huggingface-hub': '2.4.0', 'numpy': '1.26.4', 'scipy': '1.15.1', 'pandas': '2.2.3'}
Installing: ['transformers==5.18.0']
Stack check failed:
  Traceback (most recent call last):
    File "<string>", line 4, in <module>
    File "/opt/conda/lib/python3.11/site-packages/scipy/special/__init__.py", line 825, in <module>
      from . import _basic
    File "/opt/conda/lib/python3.11/site-packages/scipy/special/_basic.py", line 22, in <module>
      from ._multiufuncs import (assoc_legendre_p_all,
    File "/opt/conda/lib/python3.11/site-packages/scipy/special/_multiufuncs.py", line 142, in <module>
      sph_legendre_p = MultiUFunc(
                       ^^^^^^^^^^^
    File "/opt/conda/lib/python3.11/site-packages/scipy/special/_multiufuncs.py", line 41, in __init__
      raise ValueError("All ufuncs must have type `numpy.ufunc`."
  ValueError: All ufuncs must have type `numpy.ufunc`. Received 

In [3]:
import os, re, json, time, gc, random, bisect, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- paths (same convention as Assignment 1)
def _project_dir():
    # PROJECT_DIR if set; otherwise the notebook's own folder when it holds domain_corpus/; otherwise ~/LLM_Assignment
    if os.environ.get("PROJECT_DIR"):
        return Path(os.environ["PROJECT_DIR"]).expanduser()
    return Path.cwd() if (Path.cwd() / "domain_corpus").is_dir() else Path.home() / "LLM_Assignment"
PROJECT_DIR = _project_dir()
CORPUS_DIR  = PROJECT_DIR / "domain_corpus"          # cleaned .txt files from Assignment 1
EVAL_JSONL  = PROJECT_DIR / "eval.jsonl"             # Assignment 1 held-out evaluation pairs
TRAIN_JSONL = PROJECT_DIR / "train.jsonl"
ADAPTER_DIR = next((d for d in (PROJECT_DIR / "adapter_B_r16", PROJECT_DIR) if (d / "adapter_config.json").exists()),
                   PROJECT_DIR / "adapter_B_r16")   # Assignment 1 QLoRA adapter (its own folder, or loose in the project folder)
OUT_DIR     = PROJECT_DIR / "assignment_2b"; OUT_DIR.mkdir(exist_ok=True)

# ---- models
EMBED_MODEL  = os.environ.get("EMBED_MODEL_PATH", "BAAI/bge-small-en-v1.5")   # 512-token limit, 384-d
QUERY_PREFIX = "Represent this sentence for searching relevant passages: " if "bge" in EMBED_MODEL.lower() else ""
RERANK_MODEL = os.environ.get("RERANK_MODEL_PATH", "cross-encoder/ms-marco-MiniLM-L-6-v2")
GEN_BASE     = os.environ.get("QWEN_7B_PATH", "Qwen/Qwen2.5-7B")              # Assignment 1 base model
GEN_INSTRUCT = os.environ.get("QWEN_7B_INSTRUCT_PATH", "Qwen/Qwen2.5-7B-Instruct")  # fallback generator
JUDGE_MODEL  = os.environ.get("JUDGE_MODEL_PATH", "meta-llama/Llama-3.1-8B-Instruct")  # must differ from the generator
JUDGE_4BIT   = True                                                           # 4-bit keeps the judge small on a shared GPU

CHUNK_SIZES = [100, 400]     # the brief's text says 400 (its table header says 500); text is followed here
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "-")
import shutil
from huggingface_hub import constants as hf_constants
HF_CACHE = Path(hf_constants.HF_HUB_CACHE)
HF_CACHE.mkdir(parents=True, exist_ok=True)
def free_disk_gb():
    return shutil.disk_usage(HF_CACHE).free / 1e9
def usable_gpu_gb():
    # free memory on the (shared) GPU plus what this process has reserved but is not using
    if DEVICE != "cuda":
        return 0.0
    free, _ = torch.cuda.mem_get_info()
    return (free + torch.cuda.memory_reserved() - torch.cuda.memory_allocated()) / 1e9

for p in (CORPUS_DIR, EVAL_JSONL, TRAIN_JSONL, ADAPTER_DIR / "adapter_config.json", ADAPTER_DIR / "adapter_model.safetensors"):
    assert p.exists(), f"missing: {p} - see the folder layout in the first cell"
print("project folder:", PROJECT_DIR, "| adapter:", ADAPTER_DIR)
if DEVICE == "cuda":
    print(f"GPU memory: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB free of {torch.cuda.mem_get_info()[1]/1e9:.1f} GB (shared GPU)")
print(f"Hugging Face cache: {HF_CACHE} | free disk {free_disk_gb():.1f} GB")

device: cuda | GPU: NVIDIA A100-SXM4-80GB
project folder: /home/jovyan/Assignment_2B | adapter: /home/jovyan/Assignment_2B/adapter_B_r16
GPU memory: 84.5 GB free of 85.0 GB (shared GPU)
Hugging Face cache: /home/jovyan/.cache/huggingface/hub | free disk 13.4 GB


### Load the corpus

The Assignment 1 `.txt` files carry `<<<PAGE n>>>` markers and hard line-wraps. Both are removed (whitespace collapsed to single spaces) so that
chunks are not cut by page markers and answer spans can be matched as plain strings. The page number of every chunk is kept for reporting.

In [4]:
PAGE_RE = re.compile(r"<<<PAGE (\d+)>>>\n?")

def load_corpus(corpus_dir):
    docs = []
    for f in sorted(corpus_dir.glob("*.txt")):
        parts = PAGE_RE.split(f.read_text(encoding="utf-8"))
        text, pages = "", []                       # pages: [(start_char, page_no)]
        if len(parts) == 1:
            text = re.sub(r"\s+", " ", parts[0]).strip(); pages = [(0, 1)]
        else:
            for pno, ptxt in zip(parts[1::2], parts[2::2]):
                ptxt = re.sub(r"\s+", " ", ptxt).strip()
                if not ptxt:
                    continue
                if text:
                    text += " "
                pages.append((len(text), int(pno))); text += ptxt
        docs.append({"name": f.stem, "text": text, "pages": pages})
    return docs

DOCS = load_corpus(CORPUS_DIR)
def page_of(doc_idx, pos):
    starts = [p[0] for p in DOCS[doc_idx]["pages"]]
    return DOCS[doc_idx]["pages"][max(bisect.bisect_right(starts, pos) - 1, 0)][1]

display(pd.DataFrame([{"doc": d["name"], "characters": len(d["text"]), "pages": len(d["pages"])} for d in DOCS]))

,doc,characters,pages
0,BASEL_basel-framework-full,288293,119
1,FATF_fatf-recommendations-2012,341495,146
2,IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulati...,120008,65
3,RBI_148MD,61158,36
4,RBI_163MD,263742,149
5,RBI_169MD,196220,97
6,SEBI_1767852346757,57952,24


### The 10 evaluation queries (from Assignment 1)

P1–P3 are the three evaluation prompts of Assignment 1 Part A. The other seven are one held-out question per instrument drawn from `eval.jsonl`
with the same seed and rule as the Assignment 1 benchmark set (Block 14), so this is the same 10-question set used in Assignment 1.

In [5]:
P_QUERIES = [
    {"id": "P1", "query": "Under the RBI KYC Directions, who is the beneficial owner when the customer is a company?",
     "expected": "The natural person(s) who, alone or together, or through one or more juridical persons, has a controlling ownership interest "
                 "(ownership of / entitlement to more than 10 percent of the shares, capital or profits of the company) or who exercises control "
                 "through other means (e.g. the right to appoint the majority of directors or control management or policy decisions)."},
    {"id": "P2", "query": "What minimum haircut applies to Level 2A assets in a bank's stock of high-quality liquid assets, and what is the cap on Level 2 assets?",
     "expected": "A minimum 15 percent haircut on the current market value of each Level 2A asset; Level 2 assets (2A and 2B), after haircuts, "
                 "may comprise no more than 40 percent of the overall stock of HQLA."},
    {"id": "P3", "query": "With how many life insurers may a Corporate Agent (Life) have arrangements?",
     "expected": "A maximum of nine life insurers, to solicit, procure and service their insurance products."},
]
def read_jsonl(path):
    """Read a .jsonl (or a JSON array) file with clear errors instead of pandas' 'Expected object or value'."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Contents of {path.parent}: {sorted(os.listdir(path.parent)) if path.parent.exists() else 'folder missing'}")
    raw = path.read_text(encoding="utf-8-sig").strip()
    if not raw:
        raise ValueError(f"{path} is empty ({path.stat().st_size} bytes): re-upload it from the Assignment 1 project folder")
    if raw[0] == "[":
        return json.loads(raw)
    rows = []
    for n, line in enumerate(raw.splitlines(), 1):
        if line.strip():
            try: rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"{path.name} line {n} is not valid JSON ({e}). Starts with: {line[:80]!r}")
    return rows

eval_pool = pd.DataFrame(read_jsonl(EVAL_JSONL))
assert {"instruction", "response", "source"} <= set(eval_pool.columns), f"unexpected columns in eval.jsonl: {list(eval_pool.columns)}"
rng = random.Random(SEED)
extra = [g.iloc[rng.randrange(len(g))] for _, g in sorted(eval_pool.groupby("source"), key=lambda kv: kv[0])]
QUERIES = P_QUERIES + [{"id": f"E{i+1}", "query": r.instruction, "expected": r.response, "source": r.source} for i, r in enumerate(extra)]
QUERIES = QUERIES[:10]
assert len(QUERIES) == 10, len(QUERIES)

# one question whose answer is NOT in the corpus (Part D1 step 3) - near-domain on purpose
UNANSWERABLE = "What is the minimum paid-up capital required to set up a cryptocurrency exchange in India under the RBI Directions?"
display(pd.DataFrame(QUERIES)[["id", "query", "expected"]])

,id,query,expected
0,P1,"Under the RBI KYC Directions, who is the benef...","The natural person(s) who, alone or together, ..."
1,P2,What minimum haircut applies to Level 2A asset...,A minimum 15 percent haircut on the current ma...
2,P3,With how many life insurers may a Corporate Ag...,"A maximum of nine life insurers, to solicit, p..."
3,E1,What RSF factor is specified by the Basel Fram...,Table 1 of the Basel Framework specifies that ...
4,E2,What measures are required regarding cross-bor...,Countries must establish declaration or disclo...
5,E3,What specific declarations regarding previous ...,"Principal officers, directors, and managing pa..."
6,E4,How frequently are domestic structural liquidi...,Paragraph 35 establishes that a bank must prep...
7,E5,How must a bank proceed when local laws and re...,The bank must bring the prohibition to the not...
8,E6,How are group entities defined and classified ...,Paragraph 4(9A) defines a group entity as an e...
9,E7,Under regulation 32 of the SEBI Stock Brokers ...,Regulation 32 states that the Board may appoin...


## Part A — Dense search and chunk size (5 marks)

### Answer spans and the embedding model's input limit

For each query the **answer span** is the sentence or short passage in the corpus that answers it. The cell below proposes, for each
query, the corpus sentence that shares most content words and numbers with the expected answer, and locates it (document, page,
character range). Every proposal was reviewed against the corpus; where it was wrong, the correct sentence was set by hand
(`SPAN_OVERRIDES`). The review is summarised under the table. A chunk counts as a **hit** when it fully contains the span.

In [6]:
from transformers import AutoTokenizer
emb_tok = AutoTokenizer.from_pretrained(EMBED_MODEL, use_fast=True)
emb_tok.model_max_length = 10**9                          # we chunk by hand; silence the length warning
print("Embedding model:", EMBED_MODEL)

STOP = set("""the a an and or of to in for on by with as at from that this these those which who whom whose is are was were be been being shall may must
will can should would could not no any all each such other its their his her it they he she than then there where when under within into upon also only per more
less including include includes whether either neither both same have has had do does""".split())
cw = lambda t: {w for w in re.findall(r"[a-z]+", t.lower()) if len(w) >= 4 and w not in STOP}
nums = lambda t: set(re.findall(r"\d+(?:\.\d+)?", t))
BOUND = re.compile(r"(?<=[.;])\s+(?=[A-Z(\d‘“'\"])|\s+(?=\((?:[b-z]|ii|iii|iv|v|vi|vii|viii|ix|x)\)\s)")   # sentence ends, plus "(b)" / "(iii)" list items

def sentences(doc_idx):
    text, out, s = DOCS[doc_idx]["text"], [], 0
    for m in list(BOUND.finditer(text)) + [None]:
        e = m.start() if m else len(text)
        if 8 <= len(text[s:e].split()) <= 120:
            out.append((s, e))
        s = m.end() if m else e
    return out

SENT = {i: sentences(i) for i in range(len(DOCS))}

def suggest_spans(expected, query="", n=3):
    target_w, target_n, query_w = cw(expected), nums(expected), cw(query)
    cands = []
    for di, spans in SENT.items():
        t = DOCS[di]["text"]
        for s, e in spans:
            sw = cw(t[s:e])
            score = (len(target_w & sw) / (len(target_w) or 1) + 0.5 * len(query_w & sw) / (len(query_w) or 1)
                     + 0.5 * len(target_n & nums(t[s:e])) / (len(target_n) or 1) - 0.0005 * (e - s) / 10)   # tiny bias to the shorter sentence
            cands.append((score, (di, s, e)))
    return sorted(cands, key=lambda c: -c[0])[:n]

SPAN_PICK = {}           # e.g. {"P1": 2} = take the 2nd-ranked candidate instead of the 1st
# Spans are pre-filled after checking them against the corpus text: the automatic pick lands on the partnership-firm clause for P1, and the
# full sentences for E3 (a fit-and-proper form row) and E5 straddle a 100-token chunk boundary, so their key phrase is used instead.
# Every override was confirmed to occur verbatim in the corpus.
SPAN_OVERRIDES = {
    "P1": "Where the customer is a company, the beneficial owner is the natural person(s), who, whether acting alone or together, or through one or more juridical persons, has / have a controlling ownership interest or who exercises control through other means.",
    "P2": "A minimum 15 per cent haircut shall be applied to the current market value of each Level 2A asset",
    "P3": "A Corporate Agent (Life), may have arrangements with a maximum of nine life insurers",
    "E1": "National supervisors may specify the RSF factors based on their national circumstances Other contingent funding obligations, including products and instruments such as unconditionally revocable credit and liquidity facilities",
    "E3": "Have you ever registered or obtained license from any of the regulatory authorities",
    "E4": "A bank shall prepare domestic SLS (₹) on a daily basis and report to RBI on a fortnightly basis. Further, SLS in respect of overseas branch operations shall be reported to RBI on monthly basis.",
    "E5": "prohibit implementation of these guidelines, the bank shall bring the same to the notice of the RBI",
    "E7": "In addition to the inspection by inspecting authority, the Board may appoint a qualified auditor to inspect or investigate into the books of account or the affairs of the stock broker.",
}

# If a span cannot fit inside one fixed chunk at every compared size (it straddles a boundary), the coverage cell below switches to the
# first of these shorter key phrases that does fit at every size. Each phrase is a verbatim, unique substring of the corpus that still
# contains the fact the question asks for (a phrase without it would make a hit meaningless), and the same span is then used for both
# chunk sizes, so the comparison stays fair.
SPAN_FALLBACKS = {
    "P1": ["the beneficial owner is the natural person(s), who, whether acting alone or together, or through one or more juridical persons, has / have a controlling ownership interest",
           "Where the customer is a company, the beneficial owner is the natural person(s)",
           "has / have a controlling ownership interest or who exercises control through other means"],
    "P2": ["15 per cent haircut shall be applied to the current market value of each Level 2A asset"],
    "P3": ["may have arrangements with a maximum of nine life insurers", "a maximum of nine life insurers to solicit"],
    "E1": ["may specify the RSF factors based on their national circumstances Other contingent funding obligations"],
    "E2": ["detect the physical cross-border transportation of currency and bearer negotiable instruments, including through a declaration system",
           "Countries should have measures in place to detect the physical cross-border transportation of currency",
           "including through a declaration system and/or disclosure system"],
    "E3": ["Have you ever registered or obtained license",
           "obtained license from any of the regulatory authorities under any law such as SEBI, RBI, IRDA, PFRDA, FMC"],
    "E4": ["A bank shall prepare domestic SLS (₹) on a daily basis and report to RBI on a fortnightly basis",
           "SLS in respect of overseas branch operations shall be reported to RBI on monthly basis",
           "report to RBI on a fortnightly basis"],
    "E5": ["the bank shall bring the same to the notice of the RBI"],
    "E6": ["‘Group entity’ means an entity which is a subsidiary or joint venture or associate of a bank",
           "Entities held under NOFHC shall not be treated as group entities"],
    "E7": ["the Board may appoint a qualified auditor to inspect or investigate into the books of account or the affairs of the stock broker",
           "the Board may appoint a qualified auditor to inspect or investigate",
           "qualified auditor to inspect or investigate into the books of account or the affairs of the stock broker"],
}

def locate(text_span):
    # every verbatim occurrence of the span; a chunk holding any of them is a hit
    norm = re.sub(r"\s+", " ", text_span).strip()
    occ = [(di, m.start(), m.end()) for di, d in enumerate(DOCS) for m in re.finditer(re.escape(norm), d["text"])]
    if not occ:
        raise ValueError("override span not found verbatim in the corpus: " + norm[:80])
    return occ

SPANS, rows, ALTS = {}, [], {}
for q in QUERIES:
    cands = suggest_spans(q["expected"], q["query"]); ALTS[q["id"]] = cands
    if q["id"] in SPAN_OVERRIDES:
        occ = locate(SPAN_OVERRIDES[q["id"]]); score, how = float("nan"), "manual text"
    else:
        pick = SPAN_PICK.get(q["id"], 1); score, occ0 = cands[pick - 1]; how = f"candidate {pick}"
        occ = locate(DOCS[occ0[0]]["text"][occ0[1]:occ0[2]])
    SPANS[q["id"]] = [{"doc": d_, "start": s_, "end": e_} for d_, s_, e_ in occ]
    di, s, e = occ[0]
    rows.append({"id": q["id"], "how": how, "match": round(score, 2), "occurrences": len(occ), "document": DOCS[di]["name"], "page": page_of(di, s),
                 "span_tokens": len(emb_tok(DOCS[di]["text"][s:e], add_special_tokens=False)["input_ids"]),
                 "span": DOCS[di]["text"][s:e]})
span_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 400)
display(span_df)
print("\nOther candidates (use SPAN_PICK = {id: rank} to choose one):")
for q in QUERIES:
    for r, (sc, (di, st, en)) in enumerate(ALTS[q["id"]], 1):
        print(f"  {q['id']} #{r} ({sc:.2f}, {DOCS[di]['name']} p.{page_of(di, st)}): {DOCS[di]['text'][st:en][:170]}")
print("Check every row. A question with two parts (e.g. P2: haircut AND cap) may need the span set to the sentence holding the key fact. "
      "Spans longer than ~50 tokens often straddle a 100-token chunk boundary (checked after chunking, below).")

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Embedding model: BAAI/bge-small-en-v1.5


,id,how,match,occurrences,document,page,span_tokens,span
0,P1,manual text,NaN,1,RBI_169MD,5,51,"Where the customer is a company, the beneficial owner is the natural person(s), who, whether acting alone or together, or through one or more juridical persons, has / have a controlling ownership interest or who exercises control through other means."
1,P2,manual text,NaN,1,RBI_163MD,55,20,A minimum 15 per cent haircut shall be applied to the current market value of each Level 2A asset
2,P3,manual text,NaN,1,IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0,3,19,"A Corporate Agent (Life), may have arrangements with a maximum of nine life insurers"
3,E1,manual text,NaN,1,BASEL_basel-framework-full,929,36,"National supervisors may specify the RSF factors based on their national circumstances Other contingent funding obligations, including products and instruments such as unconditionally revocable credit and liquidity facilities"
4,E2,candidate 1,0.61,1,FATF_fatf-recommendations-2012,26,39,"Cash couriers * Countries should have measures in place to detect the physical cross-border transportation of currency and bearer negotiable instruments, including through a declaration system and/or disclosure system."
5,E3,manual text,NaN,1,IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0,27,13,Have you ever registered or obtained license from any of the regulatory authorities
6,E4,manual text,NaN,1,RBI_163MD,16,44,"A bank shall prepare domestic SLS (₹) on a daily basis and report to RBI on a fortnightly basis. Further, SLS in respect of overseas branch operations shall be reported to RBI on monthly basis."
7,E5,manual text,NaN,1,RBI_169MD,4,18,"prohibit implementation of these guidelines, the bank shall bring the same to the notice of the RBI"
8,E6,candidate 1,0.81,1,RBI_148MD,6,25,(9A) 4‘Group entity’ means an entity which is a subsidiary or joint venture or associate of a bank.
9,E7,manual text,NaN,1,SEBI_1767852346757,40,33,"In addition to the inspection by inspecting authority, the Board may appoint a qualified auditor to inspect or investigate into the books of account or the affairs of the stock broker."



Other candidates (use SPAN_PICK = {id: rank} to choose one):
  P1 #1 (1.32, RBI_169MD p.5): (b) Where the customer is a partnership firm, the beneficial owner is the natural person(s), who, whether acting alone or together, or through one or more juridical perso
  P1 #2 (1.27, RBI_169MD p.5): Explanation: For the purpose of this sub-clause- • ‘Controlling ownership interest’ means ownership of / entitlement to more than 10 percent of the shares or capital or p
  P1 #3 (1.05, RBI_169MD p.6): (d) Where the customer is a trust, the identification of beneficial owner(s) shall include identification of the author of the trust, the trustee, the beneficiaries with 
  P2 #1 (1.15, RBI_163MD p.55): Level 2A Assets (1) A minimum 15 per cent haircut shall be applied to the current market value of each Level 2A asset held in the stock.
  P2 #2 (1.14, RBI_163MD p.59): Level 2 assets shall not exceed 40 per cent of the overall stock of liquid assets after haircuts have been applied and Level 2B ass

**Span review.**
- **Checked against the corpus:** every span in the table was read and confirmed to answer its question.
- **Pre-filled by hand (6):**
  - **P1:** the company clause. The automatic pick landed on the partnership-firm clause (candidate #1), which is the wrong customer type.
  - **P2:** the 15 per cent haircut sentence (paragraph 143). The 40 per cent cap is at the end of the preceding paragraph (142), and the same 100-token chunk holds both.
  - **P3:** the Corporate Agent (Life) limit of nine life insurers.
  - **E1:** the Table 1 row for other contingent funding obligations.
  - **E4:** the daily/fortnightly and monthly reporting sentences.
  - **E7:** regulation 32(1).
- **Shortened to their key phrase (2):** the full sentences for E3 and E5 straddle a 100-token chunk boundary.
  - **E3:** the coverage check (two cells below) shortened it further, to "obtained license from any of the regulatory authorities under any law such as SEBI, RBI, IRDA, PFRDA, FMC".
  - **E5:** still straddles a boundary at 100 tokens, so it counts as a miss at that size and as a hit at 400. We left it like that rather than cut the span until it fits. An answer split across two small chunks is a real cost of 100-token chunking, not a labelling problem.
- **Automatic picks accepted (2):** E2 (FATF Recommendation 32, cash couriers) and E6 (paragraph 4(9A) definition of a group entity) are the sentences that answer their questions.

### Chunking (fixed size, no overlap, measured in embedding-model tokens)

In [7]:
def chunk_corpus(size):
    chunks = []
    for di, d in enumerate(DOCS):
        enc = emb_tok(d["text"], add_special_tokens=False, return_offsets_mapping=True)
        offs = enc["offset_mapping"]
        for i in range(0, len(offs), size):
            j = min(i + size, len(offs))
            s, e = offs[i][0], offs[j - 1][1]
            chunks.append({"id": len(chunks), "doc": di, "name": d["name"], "start": s, "end": e, "n_tokens": j - i,
                           "page": page_of(di, s), "text": d["text"][s:e]})
    return chunks

def is_hit(chunk, spans):
    # hit = the chunk fully contains (any occurrence of) the answer span
    return any(chunk["doc"] == sp["doc"] and chunk["start"] <= sp["start"] and sp["end"] <= chunk["end"] for sp in spans)

CHUNKS = {sz: chunk_corpus(sz) for sz in CHUNK_SIZES}
print({sz: len(c) for sz, c in CHUNKS.items()})

{100: 2674, 400: 672}


In [8]:
# A span that straddles a chunk boundary can never be a hit with fixed, non-overlapping chunks, which would unfairly penalise one size.
# For such a span the first shorter key phrase from SPAN_FALLBACKS that fits inside one chunk at EVERY size is used instead.
def containable(spans):
    return {sz: any(is_hit(c, spans) for c in CHUNKS[sz]) for sz in CHUNK_SIZES}

span_changes = []
for q in QUERIES:
    qid = q["id"]
    if all(containable(SPANS[qid]).values()):
        continue
    for alt in SPAN_FALLBACKS.get(qid, []):
        occ = locate(alt)
        cand = [{"doc": d_, "start": s_, "end": e_} for d_, s_, e_ in occ]
        if len(occ) == 1 and all(containable(cand).values()):
            span_changes.append({"id": qid, "replaced by": alt}); SPANS[qid] = cand
            break
if span_changes:
    print("Spans shortened because the full passage straddles a chunk boundary:")
    display(pd.DataFrame(span_changes))
cov = pd.DataFrame([{"id": q["id"], **{f"inside a {sz}-token chunk": v for sz, v in containable(SPANS[q["id"]]).items()}} for q in QUERIES])
display(cov)
straddling = cov[~cov.drop(columns="id").all(axis=1)].id.tolist()
if straddling:
    print(f"WARNING: {straddling} still straddle a chunk boundary and will count as misses at the sizes marked False. "
          "Add a shorter phrase that keeps the fact to SPAN_FALLBACKS, re-run from the span cell, and note it in the span review.")

Spans shortened because the full passage straddles a chunk boundary:


,id,replaced by
0,E3,"obtained license from any of the regulatory authorities under any law such as SEBI, RBI, IRDA, PFRDA, FMC"


,id,inside a 100-token chunk,inside a 400-token chunk
0,P1,True,True
1,P2,True,True
2,P3,True,True
3,E1,True,True
4,E2,True,True
5,E3,True,True
6,E4,True,True
7,E5,False,True
8,E6,True,True
9,E7,True,True


### A1 — Dense search (2 marks)

`build_dense_index` embeds a list of chunks (L2-normalised vectors), builds a FAISS `IndexFlatIP` (inner product on unit vectors = cosine similarity) and returns an object whose
`.search(query, k)` returns the top-k chunk ids with their scores.

In [9]:
import faiss
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
EMBED_BATCH = 64
print("Model max input length:", embedder.max_seq_length, "tokens")

class DenseIndex:
    def __init__(self, chunks, model, batch_size=EMBED_BATCH):
        self.chunks, self.model = chunks, model
        t0 = time.perf_counter()
        emb = model.encode([c["text"] for c in chunks], batch_size=batch_size, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False).astype("float32")
        self.index = faiss.IndexFlatIP(emb.shape[1]); self.index.add(emb)
        if DEVICE == "cuda": torch.cuda.synchronize()
        self.build_seconds, self.dim = time.perf_counter() - t0, emb.shape[1]

    def search(self, query, k=5):
        q = self.model.encode([QUERY_PREFIX + query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        scores, ids = self.index.search(q, k)
        return ids[0].tolist(), scores[0].tolist()

def build_dense_index(chunks):          # the one reusable function the brief asks for
    return DenseIndex(chunks, embedder)

# does each chunk size fit the embedding model? (re-tokenise the chunk text with special tokens, as the model does)
fit = []
for sz, ch in CHUNKS.items():
    mx = max(len(emb_tok(c["text"])["input_ids"]) for c in ch)
    fit.append({"chunk size": sz, "longest chunk incl. special tokens": mx, "model limit": embedder.max_seq_length,
                "fits": mx <= embedder.max_seq_length})
display(pd.DataFrame(fit))
assert all(f["fits"] for f in fit), "a chunk size exceeds the embedding limit: switch EMBED_MODEL or use a smaller size and say so below"

DENSE = {sz: build_dense_index(ch) for sz, ch in CHUNKS.items()}

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model max input length: 512 tokens


,chunk size,longest chunk incl. special tokens,model limit,fits
0,100,104,512,True
1,400,404,512,True


### A2 — Choose a chunk size (3 marks)

In [10]:
def evaluate(search_fn, k, size_chunks, warmup=True):
    """search_fn(query, k) -> list of chunk ids. Returns summary and per-query details."""
    if warmup:
        search_fn(QUERIES[0]["query"], k)                       # one warm-up run, not timed
    per, lat = [], []
    for q in QUERIES:
        span = SPANS[q["id"]]
        t0 = time.perf_counter(); ids = search_fn(q["query"], k)
        if DEVICE == "cuda": torch.cuda.synchronize()
        lat.append((time.perf_counter() - t0) * 1000)
        first = next((r for r, i in enumerate(ids, 1) if is_hit(size_chunks[i], span)), None)
        per.append({"id": q["id"], "ids": ids, "first_hit_rank": first})
    h1 = 100 * np.mean([p["first_hit_rank"] == 1 for p in per])
    h3 = 100 * np.mean([p["first_hit_rank"] is not None and p["first_hit_rank"] <= 3 for p in per])
    return {"latency_ms": float(np.mean(lat)), "hit@1": h1, "hit@3": h3}, per

a2_rows = []
for sz in CHUNK_SIZES:
    ch, di = CHUNKS[sz], DENSE[sz]
    summ, per = evaluate(lambda q, k: di.search(q, k)[0], 3, ch)
    tok3 = np.mean([sum(ch[i]["n_tokens"] for i in p["ids"][:3]) for p in per])
    a2_rows.append({"Chunk size (tokens)": sz, "Total chunks": len(ch), "Hit rate@3 (%)": summ["hit@3"],
                    "Tokens sent per query (top 3)": round(tok3, 1), "Index build (s)": round(di.build_seconds, 2)})
a2_df = pd.DataFrame(a2_rows); a2_df.to_csv(OUT_DIR / "a2_chunk_size.csv", index=False)
display(a2_df)

# choice rule: higher hit@3, then fewer tokens sent (the reasoning is given below the table)
CHOSEN_SIZE = int(a2_df.sort_values(["Hit rate@3 (%)", "Tokens sent per query (top 3)"], ascending=[False, True]).iloc[0]["Chunk size (tokens)"])
print("Auto-suggested chunk size:", CHOSEN_SIZE)

,Chunk size (tokens),Total chunks,Hit rate@3 (%),Tokens sent per query (top 3),Index build (s)
0,100,2674,60.0,300.0,1.99
1,400,672,60.0,1200.0,1.34


Auto-suggested chunk size: 100


In [11]:
# context the Assignment 1 model was trained on (to judge how much retrieved context it can handle)
q_tok = AutoTokenizer.from_pretrained(GEN_BASE)
PROMPT_A1 = "### Instruction:\n{instruction}\n\n### Response:\n"
lens = [len(q_tok(PROMPT_A1.format(instruction=r["instruction"]) + r["response"])["input_ids"]) for r in map(json.loads, open(TRAIN_JSONL, encoding="utf-8"))]
print(f"Assignment 1 training examples: mean {np.mean(lens):.0f}, max {max(lens)} tokens | base model context window: {q_tok.model_max_length if q_tok.model_max_length < 10**6 else '32,768 (Qwen2.5)'}")
print(f"Top-3 context at chunk size {CHOSEN_SIZE}: about {int(a2_df.set_index('Chunk size (tokens)').loc[CHOSEN_SIZE, 'Tokens sent per query (top 3)'])} tokens (embedding-tokenizer count)")

Assignment 1 training examples: mean 94, max 166 tokens | base model context window: 131072
Top-3 context at chunk size 100: about 300 tokens (embedding-tokenizer count)


**A2: chosen chunk size.**

We use **100 tokens** for Parts B to D:
- **Hit rate:** both sizes find the answer span in the top 3 for 6 of 10 queries (60%).
- **Context sent:** 100-token chunks send about 300 tokens per query, against 1,200 for 400-token chunks.
- **What the Assignment 1 model is used to:** its training examples were short, 94 tokens on average and 166 at most. Three 100-token chunks (about 300 tokens of context) are much closer to that than 1,200 tokens.

The cost of small chunks is that an answer can be split. E5's answer sentence straddles a 100-token boundary, and P1's definition is separated from its "Explanation" (the more-than-10-percent threshold) by a chunk boundary.

Neither size had to change for the embedding model: bge-small-en-v1.5 accepts 512 tokens, and the longest chunks are 104 and 404 tokens including special tokens.

In [12]:
FINAL_CHUNKS, FINAL_DENSE = CHUNKS[CHOSEN_SIZE], DENSE[CHOSEN_SIZE]
a1_report = pd.DataFrame([
    ("Embedding model", EMBED_MODEL),
    ("Vector size", FINAL_DENSE.dim),
    ("Similarity measure", "cosine similarity (inner product on L2-normalised vectors, FAISS IndexFlatIP)"),
    ("Batch size", EMBED_BATCH),
    ("Chunks in the final index", len(FINAL_CHUNKS)),
    (f"Index build time, chunk size {CHOSEN_SIZE}", f"{FINAL_DENSE.build_seconds:.2f} s ({DEVICE})"),
], columns=["item", "value"])
a1_report.to_csv(OUT_DIR / "a1_index_report.csv", index=False)
display(a1_report)

,item,value
0,Embedding model,BAAI/bge-small-en-v1.5
1,Vector size,384
2,Similarity measure,"cosine similarity (inner product on L2-normalised vectors, FAISS IndexFlatIP)"
3,Batch size,64
4,Chunks in the final index,2674
5,"Index build time, chunk size 100",1.99 s (cuda)


## Part B — Keyword and hybrid search (4 marks)

All three methods use the chunks chosen in Part A and return the top 5. Hybrid fuses the **top 20** of each list with Reciprocal Rank Fusion,
`RRF(d) = 1/(60 + rank_dense) + 1/(60 + rank_bm25)`, ranks starting at 1; a chunk missing from one list gets 0 from that list.
Latency is the time to return the top 5 for one query (averaged over the 10 queries, after one un-timed warm-up run); for hybrid it includes both retrievals and the fusion.

In [13]:
from rank_bm25 import BM25Okapi
bm_tok = lambda t: [w for w in re.findall(r"[a-z0-9]+", t.lower()) if w not in STOP]

class BM25Index:
    def __init__(self, chunks):
        t0 = time.perf_counter()
        self.bm25 = BM25Okapi([bm_tok(c["text"]) for c in chunks]); self.build_seconds = time.perf_counter() - t0
    def search(self, query, k=5):
        s = self.bm25.get_scores(bm_tok(query))
        ids = np.argsort(-s, kind="stable")[:k]
        return ids.tolist(), s[ids].tolist()

BM25 = BM25Index(FINAL_CHUNKS)

def rrf_fuse(dense_ids, bm25_ids, k_const=60, top=5):
    score = {}
    for rank, i in enumerate(dense_ids, 1): score[i] = score.get(i, 0) + 1 / (k_const + rank)
    for rank, i in enumerate(bm25_ids, 1):  score[i] = score.get(i, 0) + 1 / (k_const + rank)
    order = sorted(score, key=lambda i: (-score[i], dense_ids.index(i) if i in dense_ids else 10**6))
    return order[:top]

FUSE_DEPTH = 20
def dense_fn(q, k):  return FINAL_DENSE.search(q, k)[0]
def bm25_fn(q, k):   return BM25.search(q, k)[0]
def hybrid_fn(q, k): return rrf_fuse(FINAL_DENSE.search(q, FUSE_DEPTH)[0], BM25.search(q, FUSE_DEPTH)[0], top=k)

# sanity check of the fusion formula on a tiny example (ranks 1-based): d=[7,3], b=[3,9] -> 3 first (1/62 + 1/61)
assert rrf_fuse([7, 3], [3, 9], top=3) == [3, 7, 9]

b_rows, b_per = [], {}
for name, fn in [("Dense", dense_fn), ("BM25", bm25_fn), ("Hybrid (RRF)", hybrid_fn)]:
    summ, per = evaluate(fn, 5, FINAL_CHUNKS); b_per[name] = per
    b_rows.append({"Method": name, "Average query latency (ms)": round(summ["latency_ms"], 2),
                   "Hit rate@1 (%)": summ["hit@1"], "Hit rate@3 (%)": summ["hit@3"]})
b_df = pd.DataFrame(b_rows); b_df.to_csv(OUT_DIR / "b_search_comparison.csv", index=False)
display(b_df)

# which query helps explain the difference? rank of the first hit per method (blank = not in top 5)
detail = pd.DataFrame({"id": [q["id"] for q in QUERIES], **{m: [p["first_hit_rank"] for p in per] for m, per in b_per.items()}})
display(detail)

,Method,Average query latency (ms),Hit rate@1 (%),Hit rate@3 (%)
0,Dense,7.69,40.0,60.0
1,BM25,5.49,30.0,50.0
2,Hybrid (RRF),12.08,40.0,50.0


,id,Dense,BM25,Hybrid (RRF)
0,P1,1.0,3.0,1.0
1,P2,4.0,NaN,5.0
2,P3,1.0,1.0,1.0
3,E1,3.0,1.0,2.0
4,E2,1.0,2.0,1.0
5,E3,NaN,NaN,NaN
6,E4,5.0,NaN,4.0
7,E5,NaN,NaN,NaN
8,E6,1.0,NaN,NaN
9,E7,2.0,1.0,1.0


**B4: best search method.**

**Dense search works best for our queries:** hit rate@1 40% and hit rate@3 60%, against 30% and 50% for BM25, and 40% and 50% for the hybrid. Dense costs 7.7 ms per query; BM25 is faster (5.5 ms) but less accurate.

- **Where dense wins, E6** ("How are group entities defined …"): dense ranks the definition chunk 1st, BM25 does not return it in its top 5, and the hybrid loses it too. BM25 matches surface words: the question says "group entities … defined and classified", while the definition says "‘Group entity’ means …". Without stemming, "entities" does not match "entity", and the remaining words occur in many RBI chunks.
- **Where BM25 wins, E1:** exact terms such as "RSF factor" and "Table 1" rank BM25's hit 1st, against 3rd for dense.

With only 10 queries, RRF did not lift the hybrid above dense: chunks that both lists rank moderately pushed out chunks that only one list ranks highly.

## Part C — Reranking (4 marks)

The best Part B method (highest hit@3, then hit@1) retrieves the **top 10**; a cross-encoder re-scores every (query, chunk) pair and the top 3 are kept.

In [14]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)
METHODS = {"Dense": dense_fn, "BM25": bm25_fn, "Hybrid (RRF)": hybrid_fn}
BEST_METHOD = b_df.sort_values(["Hit rate@3 (%)", "Hit rate@1 (%)"], ascending=False).iloc[0]["Method"]
print("Best Part B method:", BEST_METHOD)
retrieve10 = lambda q: METHODS[BEST_METHOD](q, 10)

def rerank(query, ids, keep=3):
    scores = reranker.predict([(query, FINAL_CHUNKS[i]["text"]) for i in ids], batch_size=16, show_progress_bar=False)
    order = np.argsort(-np.asarray(scores), kind="stable")
    return [ids[j] for j in order[:keep]]

rerank(QUERIES[0]["query"], retrieve10(QUERIES[0]["query"]))          # warm-up
C = []
for q in QUERIES:
    ids10 = retrieve10(q["query"])
    t0 = time.perf_counter(); after = rerank(q["query"], ids10)
    if DEVICE == "cuda": torch.cuda.synchronize()
    C.append({"id": q["id"], "before": ids10[:3], "after": after, "ms": (time.perf_counter() - t0) * 1000})

def first_hit(qid, ids):
    return next((r for r, i in enumerate(ids, 1) if is_hit(FINAL_CHUNKS[i], SPANS[qid])), None)
hit1_before = 100 * np.mean([first_hit(c["id"], c["before"]) == 1 for c in C])
hit1_after  = 100 * np.mean([first_hit(c["id"], c["after"]) == 1 for c in C])
rank_change = 100 * np.mean([c["before"][0] != c["after"][0] for c in C])
c_df = pd.DataFrame([("Reranking time per query (ms)", round(np.mean([c["ms"] for c in C]), 1)),
                     ("Rank change rate (%)", rank_change),
                     ("Hit rate@1 before reranking (from Part B)", hit1_before),
                     ("Hit rate@1 after reranking", hit1_after)], columns=["Measure", "Value"])
c_df.to_csv(OUT_DIR / "c_rerank_table.csv", index=False)
display(c_df)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Best Part B method: Dense


,Measure,Value
0,Reranking time per query (ms),8.0
1,Rank change rate (%),60.0
2,Hit rate@1 before reranking (from Part B),40.0
3,Hit rate@1 after reranking,50.0


### C2 — Top 3 before and after reranking, per query (✓ = chunk contains the answer span)

In [15]:
def fmt(ids, qid):
    return "<br>".join(f"{'✓' if is_hit(FINAL_CHUNKS[i], SPANS[qid]) else '✗'} #{i} ({FINAL_CHUNKS[i]['name']}, p.{FINAL_CHUNKS[i]['page']}): "
                       f"{FINAL_CHUNKS[i]['text'][:110].replace('|', '/')}…" for i in ids)
md_rows = ["| Query | Top 3 before | Top 3 after | Top-1 changed |", "|---|---|---|---|"]
for c in C:
    md_rows.append(f"| {c['id']} | {fmt(c['before'], c['id'])} | {fmt(c['after'], c['id'])} | {'yes' if c['before'][0] != c['after'][0] else 'no'} |")
display(Markdown("\n".join(md_rows)))
pd.DataFrame([{"id": c["id"], "before": c["before"], "after": c["after"]} for c in C]).to_csv(OUT_DIR / "c_before_after.csv", index=False)

| Query | Top 3 before | Top 3 after | Top-1 changed |
|---|---|---|---|
| P1 | ✓ #2161 (RBI_169MD, p.5): means the process as defined under sub-section (c) of section 2 of the Aadhaar (Targeted Delivery of Financial…<br>✗ #2337 (RBI_169MD, p.47): to a beneficial owner, or where such funds are co-mingled at the level of the bank, the bank shall look for th…<br>✗ #879 (FATF_fatf-recommendations-2012, p.69): customer or the owner of the controlling interest is a company listed on a stock exchange and subject to discl… | ✓ #2161 (RBI_169MD, p.5): means the process as defined under sub-section (c) of section 2 of the Aadhaar (Targeted Delivery of Financial…<br>✗ #2337 (RBI_169MD, p.47): to a beneficial owner, or where such funds are co-mingled at the level of the bank, the bank shall look for th…<br>✗ #2316 (RBI_169MD, p.43): that Beneficial Ownership (BO) information available with them is accurate and shall update the same, if requi… | no |
| P2 | ✗ #159 (BASEL_basel-framework-full, p.848): ) to hold additional Level 2A assets in the stock of HQLA. These additional Level 2A assets must be subject to…<br>✗ #163 (BASEL_basel-framework-full, p.848): haircut) on the value of the Level 2A assets if the stress test results suggest that the minimum haircut impos…<br>✗ #222 (BASEL_basel-framework-full, p.859): . ensure that they observe the higher requirements; the application of higher haircuts to additional Level 2A … | ✗ #159 (BASEL_basel-framework-full, p.848): ) to hold additional Level 2A assets in the stock of HQLA. These additional Level 2A assets must be subject to…<br>✗ #222 (BASEL_basel-framework-full, p.859): . ensure that they observe the higher requirements; the application of higher haircuts to additional Level 2A …<br>✓ #1791 (RBI_163MD, p.55): haircuts, shall comprise no more than 40 per cent of the overall stock of HQLA. The portfolio of Level 2 asset… | no |
| P3 | ✓ #1230 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.3): Agents for the purpose of soliciting, procuring and servicing of Insurance business of life insurers, general …<br>✗ #1232 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.3): d) In the case of Corporate Agent (Composite), the conditions as specified in clauses (a) to (c) shall apply p…<br>✗ #1281 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.14): of such arrangement shall be for one year; b)While entering into such arrangements, no corporate agent shall p… | ✓ #1230 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.3): Agents for the purpose of soliciting, procuring and servicing of Insurance business of life insurers, general …<br>✗ #1232 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.3): d) In the case of Corporate Agent (Composite), the conditions as specified in clauses (a) to (c) shall apply p…<br>✗ #1231 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.3): icit, procure and service their insurance products. Further, the Corporate Agent (General) shall solicit, proc… | no |
| E1 | ✗ #580 (BASEL_basel-framework-full, p.934): categories and associated RSF factors Table 3 RSF factor RSF category 5% of the currently undrawn portion Irre…<br>✗ #498 (BASEL_basel-framework-full, p.919): F30.17. The RSF factors assigned to various types of assets are intended to approximate the amount of a partic…<br>✓ #562 (BASEL_basel-framework-full, p.929): category and their associated RSF factor. Off-balance sheet categories and associated RSF factors Table 1 RSF … | ✓ #562 (BASEL_basel-framework-full, p.929): category and their associated RSF factor. Off-balance sheet categories and associated RSF factors Table 1 RSF …<br>✗ #580 (BASEL_basel-framework-full, p.934): categories and associated RSF factors Table 3 RSF factor RSF category 5% of the currently undrawn portion Irre…<br>✗ #2069 (RBI_163MD, p.143): Balance Sheet Assets Associated RSF Factor (i) Irrevocable and conditionally revocable credit and liquidity fa… | yes |
| E2 | ✓ #667 (FATF_fatf-recommendations-2012, p.26): Cash couriers * Countries should have measures in place to detect the physical cross-border transportation of …<br>✗ #1137 (FATF_fatf-recommendations-2012, p.129): wide range of services, including cash management (e.g. interest-bearing accounts in a variety of currencies),…<br>✗ #1070 (FATF_fatf-recommendations-2012, p.115): 32 (CASH COURIERS) A. OBJECTIVES 1. Recommendation 32 was developed with the objective of ensuring that terror… | ✓ #667 (FATF_fatf-recommendations-2012, p.26): Cash couriers * Countries should have measures in place to detect the physical cross-border transportation of …<br>✗ #1070 (FATF_fatf-recommendations-2012, p.115): 32 (CASH COURIERS) A. OBJECTIVES 1. Recommendation 32 was developed with the objective of ensuring that terror…<br>✗ #1086 (FATF_fatf-recommendations-2012, p.117): . This includes failing to make a disclosure as required. Glossary of specific terms used in this Recommendati… | no |
| E3 | ✗ #1450 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.61): rketer of the above named corporate agent as per IRDAI(Registration of Corporate Agents) Regulations, 2015. Pl…<br>✗ #1430 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.54): -------corporate agent is hereby renewed under Regulation 11 of the IRDAI(Registration of Corporate Agents) Re…<br>✗ #1332 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.29): as required in the FORM A. b) Remittance of requisite fee by electronic mode as prescribed under Regulation 17… | ✗ #1332 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.29): as required in the FORM A. b) Remittance of requisite fee by electronic mode as prescribed under Regulation 17…<br>✗ #1450 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.61): rketer of the above named corporate agent as per IRDAI(Registration of Corporate Agents) Regulations, 2015. Pl…<br>✗ #1335 (IRDAI_IRDAI-Regn-of-Corporate-Agents--Regulations-2015_0, p.29): The additional documents will be advised based on, pattern of shareholding, any other matter required as per I… | yes |
| E4 | ✗ #1990 (RBI_163MD, p.102): to be used, bucketing, behavioural studies and validation process. All material assumptions, regardless of the…<br>✗ #1992 (RBI_163MD, p.103): specified in the table below. Further, bank shall refer Reserve Bank of India (Filing of Supervisory Returns) …<br>✗ #1991 (RBI_163MD, p.103): bank shall report the liquidity return comprising the SLS in five parts, viz., (1) Domestic currency for India… | ✗ #1992 (RBI_163MD, p.103): specified in the table below. Further, bank shall refer Reserve Bank of India (Filing of Supervisory Returns) …<br>✗ #1991 (RBI_163MD, p.103): bank shall report the liquidity return comprising the SLS in five parts, viz., (1) Domestic currency for India…<br>✓ #1634 (RBI_163MD, p.16): assets, liabilities, and OBS items. The difference between cash inflows and outflows in each time period shall… | yes |
| E5 | ✗ #2151 (RBI_169MD, p.1): RESERVE BANK OF INDIA RBI/DOR/2025-26/169 DOR.AML.REC.No.88/14.01.002/2025-26 November 28, 2025 Reserve Bank o…<br>✗ #1599 (RBI_163MD, p.6): satisfied that it is necessary and expedient in the public interest so to do, hereby, issues the Directions he…<br>✗ #1507 (RBI_148MD, p.12): 2025, vide Reserve Bank of India (Commercial Banks- Undertaking of Financial Services) Amendment Directions, 2… | ✗ #2157 (RBI_169MD, p.4): the State Bank of India, as defined respectively under clauses (c), (da), and (nc) of Section 5 of the Banking…<br>✗ #1507 (RBI_148MD, p.12): 2025, vide Reserve Bank of India (Commercial Banks- Undertaking of Financial Services) Amendment Directions, 2…<br>✗ #1599 (RBI_163MD, p.6): satisfied that it is necessary and expedient in the public interest so to do, hereby, issues the Directions he… | yes |
| E6 | ✓ #1482 (RBI_148MD, p.5): RBI from time to time. (9) ‘Government Securities’ shall have the same meaning as defined in the Government Se…<br>✗ #1836 (RBI_163MD, p.66): Bank of India (Commercial Banks – Prudential Norms on Capital Adequacy) Directions, 2025, for an indicative li…<br>✗ #1470 (RBI_148MD, p.1): RESERVE BANK OF INDIA 1 RBI/DOR/2025-26/148 DOR.AUT.REC.No.67/24-01-041/2025-26 November 28, 2025 Reserve Bank… | ✗ #1836 (RBI_163MD, p.66): Bank of India (Commercial Banks – Prudential Norms on Capital Adequacy) Directions, 2025, for an indicative li…<br>✗ #1507 (RBI_148MD, p.12): 2025, vide Reserve Bank of India (Commercial Banks- Undertaking of Financial Services) Amendment Directions, 2…<br>✗ #1471 (RBI_148MD, p.3): Reserve Bank of India (‘RBI’) in this regard, RBI being satisfied that it is necessary and expedient in the pu… | yes |
| E7 | ✗ #2616 (SEBI_1767852346757, p.38): (4) The complaints under this regulation against the Board of Directors including those against the managing d…<br>✓ #2622 (SEBI_1767852346757, p.39): Action on inspection or investigation report. 31. The Board shall after consideration of inspection or investi…<br>✗ #2597 (SEBI_1767852346757, p.34): of compliance officer. 17. (1) Every stock broker shall appoint a compliance officer who shall be responsible … | ✓ #2622 (SEBI_1767852346757, p.39): Action on inspection or investigation report. 31. The Board shall after consideration of inspection or investi…<br>✗ #2616 (SEBI_1767852346757, p.38): (4) The complaints under this regulation against the Board of Directors including those against the managing d…<br>✗ #2597 (SEBI_1767852346757, p.34): of compliance officer. 17. (1) Every stock broker shall appoint a compliance officer who shall be responsible … | yes |

**C4: did reranking help?**

**Yes.** Reranking the dense top 10 with the cross-encoder raised hit rate@1 from **40% to 50%** for only **8.0 ms per query** on the A100. It changed the first result for 6 of 10 queries.

- **E1:** the Table 1 chunk moved from 3rd to 1st.
- **P2:** the RBI chunk holding the answer entered the top 3; it was outside the dense top 3.
- **E3:** a miss before and after. Reranking can only reorder the 10 chunks it is given, so it cannot fix a retrieval miss.

At 8 ms per query the accuracy gain is clearly worth keeping.

## Part D — Generate and check answers (7 marks)

### D1 — Grounded generation (3 marks)

**Generator.** The Assignment 1 model is `Qwen2.5-7B` + QLoRA adapter `adapter_B_r16`, trained on short `### Instruction / ### Response`
pairs (not on multi-chunk prompts, and not on citing chunks or replying "Not found"). It is run first. Because it did not follow the
citation and "Not found" rules (see its compliance line and the example below), the generator was switched to the instruct model
`Qwen2.5-7B-Instruct`, as the brief allows.

In [16]:
RULES = """Answer the question using ONLY the numbered context chunks below.
Rules:
1. Use only information stated in the chunks. Do not use any outside knowledge.
2. Cite the number of every chunk you used in square brackets after the statement, for example [2].
3. If the chunks do not contain the answer, reply exactly: Not found in the documents.
4. Answer in 1-3 sentences. Keep numbers, percentages, time limits and defined terms exactly as they appear in the chunks."""

def rag_instruction(query, chunk_ids):
    ctx = "\n\n".join(f"[{n}] {FINAL_CHUNKS[i]['text']}" for n, i in enumerate(chunk_ids, 1))
    return f"{RULES}\n\nContext chunks:\n{ctx}\n\nQuestion: {query}"

print("=== PROMPT TEMPLATE (shown for the report) ===")
print(rag_instruction("<question>", [0, 1, 2]).replace(FINAL_CHUNKS[0]["text"], "<chunk 1 text>").replace(FINAL_CHUNKS[1]["text"], "<chunk 2 text>").replace(FINAL_CHUNKS[2]["text"], "<chunk 3 text>"))

=== PROMPT TEMPLATE (shown for the report) ===
Answer the question using ONLY the numbered context chunks below.
Rules:
1. Use only information stated in the chunks. Do not use any outside knowledge.
2. Cite the number of every chunk you used in square brackets after the statement, for example [2].
3. If the chunks do not contain the answer, reply exactly: Not found in the documents.
4. Answer in 1-3 sentences. Keep numbers, percentages, time limits and defined terms exactly as they appear in the chunks.

Context chunks:
[1] <chunk 1 text>

[2] <chunk 2 text>

[3] <chunk 3 text>

Question: <question>


In [17]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
# bf16 needs an Ampere+ GPU (A100 / L40S); a Colab T4 is Turing, so use float16 there
COMPUTE_DTYPE = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16
BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True)
DMAP = {"": 0} if DEVICE == "cuda" else {"": "cpu"}

def free_gpu():
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

import dataclasses, tempfile
from huggingface_hub import HfApi, scan_cache_dir
from peft import LoraConfig

def repo_size_gb(repo):
    # download size of the weights still missing from the local cache (0 if cached or a local folder)
    if Path(repo).exists():
        return 0.0
    try:
        cached = {r.repo_id for r in scan_cache_dir(HF_CACHE).repos}
        if repo in cached:
            return 0.0
        info = HfApi().model_info(repo, files_metadata=True)
        return sum((s.size or 0) for s in info.siblings if s.rfilename.endswith((".safetensors", ".bin"))) / 1e9
    except Exception:
        return float("nan")                          # unknown (offline / gated): let from_pretrained report the real error

def release_cached(repo):
    # remove one model this notebook has finished with from the Hugging Face cache, to make room for the next one
    for r in scan_cache_dir(HF_CACHE).repos:
        if r.repo_id == repo:
            scan_cache_dir(HF_CACHE).delete_revisions(*[rev.commit_hash for rev in r.revisions]).execute()
            print(f"freed {r.size_on_disk/1e9:.1f} GB of disk: removed {repo} from the Hugging Face cache")

def check_resources(repo, vram_gb, finished_with=()):
    need = repo_size_gb(repo)
    if need == need and need > free_disk_gb() - 1.0:                 # (need == need) is False for NaN
        for done in finished_with:
            release_cached(done)
    if need == need and need > free_disk_gb() - 1.0:
        raise OSError(f"{repo}: needs {need:.1f} GB of disk, {free_disk_gb():.1f} GB free at {HF_CACHE}")
    if DEVICE == "cuda" and usable_gpu_gb() < vram_gb:
        raise torch.cuda.OutOfMemoryError(f"{repo}: needs ~{vram_gb:.1f} GB of GPU memory, {usable_gpu_gb():.1f} GB usable on the shared GPU")

def adapter_dir_for_this_peft(adapter_dir):
    # The adapter was saved by a newer PEFT, whose adapter_config.json has keys older PEFT versions reject.
    # Keep only the keys this PEFT's LoraConfig defines (the dropped ones are unused options set to their defaults).
    cfg = json.loads((adapter_dir / "adapter_config.json").read_text())
    known = {f.name for f in dataclasses.fields(LoraConfig)}
    dropped = sorted(k for k in cfg if k not in known)
    if not dropped:
        return adapter_dir
    tmp = Path(tempfile.mkdtemp(prefix="adapter_"))
    (tmp / "adapter_config.json").write_text(json.dumps({k: v for k, v in cfg.items() if k in known}, indent=1))
    (tmp / "adapter_model.safetensors").symlink_to((adapter_dir / "adapter_model.safetensors").resolve())
    print(f"adapter_config.json: ignored {len(dropped)} keys unknown to this PEFT version: {dropped}")
    return tmp

def load_generator(mode):
    if mode == "a1_adapter":
        check_resources(GEN_BASE, vram_gb=6.5)
        tok = AutoTokenizer.from_pretrained(GEN_BASE)
        base = AutoModelForCausalLM.from_pretrained(GEN_BASE, quantization_config=BNB, device_map=DMAP)
        model = PeftModel.from_pretrained(base, adapter_dir_for_this_peft(ADAPTER_DIR)).eval()
        label = f"Assignment 1 model: {GEN_BASE} + QLoRA adapter {ADAPTER_DIR.name}"
        render = lambda ins: PROMPT_A1.format(instruction=ins)
        stop = [tok.convert_tokens_to_ids("<|endoftext|>")]
    else:
        check_resources(GEN_INSTRUCT, vram_gb=6.5, finished_with=[GEN_BASE] if "a1_adapter" in GEN else [])
        tok = AutoTokenizer.from_pretrained(GEN_INSTRUCT)
        model = AutoModelForCausalLM.from_pretrained(GEN_INSTRUCT, quantization_config=BNB, device_map=DMAP).eval()
        label = f"Fallback instruct model: {GEN_INSTRUCT}"
        render = lambda ins: tok.apply_chat_template([{"role": "user", "content": ins}], tokenize=False, add_generation_prompt=True)
        stop = [tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")]
    return tok, model, render, stop, label

@torch.inference_mode()
def generate_text(tok, model, render, stop, instruction, max_new_tokens=160):
    enc = tok(render(instruction), return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, eos_token_id=stop, pad_token_id=stop[0])
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# final top-3 per query = reranked top 3 from Part C
TOP3 = {c["id"]: c["after"] for c in C}
UN_IDS = rerank(UNANSWERABLE, retrieve10(UNANSWERABLE))

GEN = {}      # mode -> {"label", "answers": {id: text}, "unanswerable": text}
def run_generation(mode):
    tok, model, render, stop, label = load_generator(mode)
    answers = {q["id"]: generate_text(tok, model, render, stop, rag_instruction(q["query"], TOP3[q["id"]])) for q in QUERIES}
    un = generate_text(tok, model, render, stop, rag_instruction(UNANSWERABLE, UN_IDS))
    del model; free_gpu()
    GEN[mode] = {"label": label, "answers": answers, "unanswerable": un}
    json.dump(GEN[mode], open(OUT_DIR / f"d1_answers_{mode}.json", "w"), indent=1)

CITE = re.compile(r"\[\s*[1-3]\s*\]")
NOTFOUND = lambda t: "not found in the documents" in t.lower()
def compliance(mode):
    a = GEN[mode]["answers"]
    return {"mode": mode, "answers_with_citation": f"{sum(bool(CITE.search(t)) for t in a.values())}/10",
            "answers_with_citation_share": np.mean([bool(CITE.search(t)) for t in a.values()]),
            "unanswerable_says_not_found": NOTFOUND(GEN[mode]["unanswerable"])}

run_generation("a1_adapter")
comp = compliance("a1_adapter"); print(comp)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{'mode': 'a1_adapter', 'answers_with_citation': '6/10', 'answers_with_citation_share': 0.6, 'unanswerable_says_not_found': False}


In [18]:
FORCE_MODE = None        # set to "a1_adapter" or "instruct" to override
adapter_ok = comp["answers_with_citation_share"] >= 0.8 and comp["unanswerable_says_not_found"]
if FORCE_MODE: FINAL_MODE = FORCE_MODE
else:          FINAL_MODE = "a1_adapter" if adapter_ok else "instruct"
if FINAL_MODE == "instruct" and "instruct" not in GEN:
    run_generation("instruct"); print(compliance("instruct"))
GENERATOR_USED = GEN[FINAL_MODE]["label"]
print("Generator used for Part D:", GENERATOR_USED)

if FINAL_MODE == "instruct":
    # keep one concrete example of the Assignment 1 model's problem, as the brief asks
    bad = next((q for q in QUERIES if not CITE.search(GEN["a1_adapter"]["answers"][q["id"]])), QUERIES[0])
    display(Markdown(f"**Problem seen with the Assignment 1 model** — it did not follow the citation / 'Not found' rules. Example, {bad['id']}: *{bad['query']}*\n\n"
                     f"> {GEN['a1_adapter']['answers'][bad['id']][:600]}\n\n"
                     f"On the unanswerable query it said: > {GEN['a1_adapter']['unanswerable'][:300]}"))

freed 15.2 GB of disk: removed Qwen/Qwen2.5-7B from the Hugging Face cache


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

{'mode': 'instruct', 'answers_with_citation': '7/10', 'answers_with_citation_share': 0.7, 'unanswerable_says_not_found': True}
Generator used for Part D: Fallback instruct model: Qwen/Qwen2.5-7B-Instruct


**Problem seen with the Assignment 1 model** — it did not follow the citation / 'Not found' rules. Example, E1: *What RSF factor is specified by the Basel Framework for other contingent funding obligations in Table 1?*

> According to Table 1, the RSF factor for other contingent funding obligations is 5% of the currently undrawn portion.

On the unanswerable query it said: > The minimum paid-up capital required to set up a cryptocurrency exchange in India under the RBI Directions is ₹500 crore. This is specified in chunk [2], which outlines the conditions for banks to participate in the currency futures market as trading or clearing members.

**Answers to all 10 queries** (reranked top-3 chunks supplied, numbered [1]–[3]):

In [19]:
ANS = GEN[FINAL_MODE]["answers"]
d1_df = pd.DataFrame([{"id": q["id"], "query": q["query"], "chunks (ids)": TOP3[q["id"]], "answer": ANS[q["id"]]} for q in QUERIES])
d1_df.to_csv(OUT_DIR / "d1_answers.csv", index=False)
for q in QUERIES:
    display(Markdown(f"**{q['id']}. {q['query']}**  \nchunks: {TOP3[q['id']]}  \n> {ANS[q['id']]}"))

**P1. Under the RBI KYC Directions, who is the beneficial owner when the customer is a company?**  
chunks: [2161, 2337, 2316]  
> The beneficial owner is the natural person(s) who, whether acting alone or together, or through one or more juridical persons, has/have a controlling ownership interest or who exercises control through other means [1].

**P2. What minimum haircut applies to Level 2A assets in a bank's stock of high-quality liquid assets, and what is the cap on Level 2 assets?**  
chunks: [159, 222, 1791]  
> The minimum haircut applicable to Level 2A assets in a bank's stock of high-quality liquid assets is 15%, and the cap on Level 2 assets is no more than 40% of the overall stock of HQLA [3].

**P3. With how many life insurers may a Corporate Agent (Life) have arrangements?**  
chunks: [1230, 1232, 1231]  
> A Corporate Agent (Life) may have arrangements with a maximum of nine life insurers [1(a)].

**E1. What RSF factor is specified by the Basel Framework for other contingent funding obligations in Table 1?**  
chunks: [562, 580, 2069]  
> Not found in the documents.

**E2. What measures are required regarding cross-border currency transportation under the FATF cash couriers guidelines?**  
chunks: [667, 1070, 1086]  
> Countries are required to have measures in place to detect the physical cross-border transportation of currency and bearer negotiable instruments, which can include a declaration or disclosure system. Additionally, competent authorities must have the legal authority to stop or restrain such items if they are suspected to be related to certain criminal activities [1]. These measures aim to prevent the financing of terrorist activities and money laundering through such transportation [2].

**E3. What specific declarations regarding previous regulatory registrations must key individuals submit under the IRDAI (Registration of Corporate Agent) Regulations, 2015?**  
chunks: [1332, 1450, 1335]  
> Key individuals must submit a declaration of Fit & Proper criteria from Principal Officer, Directors, and Managing Partners (separately for each person) as part of the application, according to the Fit and Proper Form as per annexure 1 [1].

**E4. How frequently are domestic structural liquidity statements and overseas branch reports required to be submitted to the Reserve Bank of India?**  
chunks: [1992, 1991, 1634]  
> Domestic structural liquidity statements (SLS) are required to be reported on a fortnightly basis, while overseas branch operations reports are submitted on a monthly basis to the Reserve Bank of India [2][3].

**E5. How must a bank proceed when local laws and regulations abroad prohibit the implementation of the Reserve Bank of India (Commercial Banks – Know Your Customer) Directions, 2025?**  
chunks: [2157, 1507, 1599]  
> When local laws and regulations abroad prohibit the implementation of the Reserve Bank of India (Commercial Banks – Asset Liability Management) Directions, 2025, the bank shall bring the same to the notice of the Reserve Bank [1].

**E6. How are group entities defined and classified under the Reserve Bank of India (Commercial Banks – Undertaking of Financial Services) Directions, 2025?**  
chunks: [1836, 1507, 1471]  
> Not found in the documents.

**E7. Under regulation 32 of the SEBI Stock Brokers Regulations, how is the Board empowered to handle auditor appointment for a stock broker?**  
chunks: [2622, 2616, 2597]  
> Under regulation 32(1) of the SEBI Stock Brokers Regulations, the Board may appoint a qualified auditor to inspect or investigate into the books of account or the affairs of the stock broker in addition to inspections by the inspecting authority [1].

**D1 step 3 — a query whose answer is not in the corpus.** The top chunks retrieved for it (printed below) do not answer it.

In [20]:
print("Query:", UNANSWERABLE)
for n, i in enumerate(UN_IDS, 1):
    print(f"  [{n}] ({FINAL_CHUNKS[i]['name']}, p.{FINAL_CHUNKS[i]['page']}) {FINAL_CHUNKS[i]['text'][:160]}…")
print("\nModel answer:", GEN[FINAL_MODE]["unanswerable"])
print("Says 'Not found in the documents':", NOTFOUND(GEN[FINAL_MODE]["unanswerable"]))

Query: What is the minimum paid-up capital required to set up a cryptocurrency exchange in India under the RBI Directions?
  [1] (RBI_163MD, p.54) standardised approach for credit risk. (2) Overnight balances held with RBI under Standing Deposit Facility (SDF). (3) Government securities in excess of the mi…
  [2] (RBI_148MD, p.27) meeting all of the following conditions: (1) It has a minimum net worth of ₹500 crore; (2) It has the minimum prescribed capital (including Capital Conservation…
  [3] (RBI_148MD, p.31) 6, consequent to Reserve Bank of India (Commercial Banks - Governance) Amendment Directions, 2026 / silver by sending prior intimation to the Department of Regu…

Model answer: Not found in the documents.
Says 'Not found in the documents': True


### D2 — LLM judge (4 marks)

**Judge model:** a different model from the generator, preferably from a different model family, taken from this list in order (the
cell output names the model used and why earlier candidates were skipped):
1. `meta-llama/Llama-3.1-8B-Instruct` (Meta; gated, skipped here because the account has no access);
2. `ibm-granite/granite-3.3-8b-instruct` (IBM; open): **used, in bfloat16**;
3. `Qwen/Qwen2.5-14B-Instruct`, then `microsoft/Phi-3.5-mini-instruct` (fallbacks, not needed).

Decoding is greedy (`do_sample=False`, equivalent to temperature 0), so the verdict is the same on every run. One prompt checks two
separate things per answer: **CORRECT** (against the expected answer from Assignment 1) and **SUPPORTED** (against the answer's 3
chunks). Labels are defined exactly, a "Not found" reply is handled explicitly, and the output format is fixed so that code can parse
it; a reply that breaks the format is asked once more with an explicit format reminder.

In [21]:
JUDGE_TEMPLATE = """You are a strict examiner marking one answer from a question-answering system. Judge only on the material given below.

QUESTION:
<<Q>>

EXPECTED ANSWER (ground truth):
<<EXP>>

RETRIEVED CHUNKS the system was given:
<<CHUNKS>>

SYSTEM ANSWER:
<<ANS>>

Make two separate checks.

CHECK 1 - CORRECT. Compare the SYSTEM ANSWER with the EXPECTED ANSWER only (ignore the chunks for this check). Use exactly one label:
- CORRECT: it states every key fact of the expected answer (numbers, named entities, conditions) and contradicts nothing in it.
- PARTLY CORRECT: it states some but not all of the key facts, or includes a minor inaccuracy, but does not contradict the expected answer.
- WRONG: it misses the main fact, contradicts the expected answer, or is irrelevant.
If the system answer is "Not found in the documents", label it WRONG for this check, because the expected answer exists.

CHECK 2 - SUPPORTED. Answer YES only if EVERY statement in the SYSTEM ANSWER can be found in the RETRIEVED CHUNKS. Answer NO if any statement is missing from the chunks or goes beyond them.
A reply of "Not found in the documents" makes no claim, so it is SUPPORTED: YES.

Reply in exactly this format, one item per line, and nothing else:
CORRECT: <CORRECT or PARTLY CORRECT or WRONG>
SUPPORTED: <YES or NO>
REASON: <one sentence explaining both verdicts>
UNSUPPORTED: <a short quote of any unsupported statement, or NONE>"""

def judge_prompt(q, expected, chunk_ids, answer):
    chunks = "\n\n".join(f"[{n}] {FINAL_CHUNKS[i]['text']}" for n, i in enumerate(chunk_ids, 1))
    return (JUDGE_TEMPLATE.replace("<<Q>>", q).replace("<<EXP>>", expected).replace("<<CHUNKS>>", chunks).replace("<<ANS>>", answer))

print(JUDGE_TEMPLATE)

You are a strict examiner marking one answer from a question-answering system. Judge only on the material given below.

QUESTION:
<<Q>>

EXPECTED ANSWER (ground truth):
<<EXP>>

RETRIEVED CHUNKS the system was given:
<<CHUNKS>>

SYSTEM ANSWER:
<<ANS>>

Make two separate checks.

CHECK 1 - CORRECT. Compare the SYSTEM ANSWER with the EXPECTED ANSWER only (ignore the chunks for this check). Use exactly one label:
- CORRECT: it states every key fact of the expected answer (numbers, named entities, conditions) and contradicts nothing in it.
- PARTLY CORRECT: it states some but not all of the key facts, or includes a minor inaccuracy, but does not contradict the expected answer.
- WRONG: it misses the main fact, contradicts the expected answer, or is irrelevant.
If the system answer is "Not found in the documents", label it WRONG for this check, because the expected answer exists.

CHECK 2 - SUPPORTED. Answer YES only if EVERY statement in the SYSTEM ANSWER can be found in the RETRIEVED CHUN

In [29]:
# The judge must be a different model from the generator, and preferably a different model family (a model tends to agree
# with text written the way it would write it). Candidates in order; each is skipped with the reason printed if it is gated,
# missing, or does not fit the free disk / GPU memory:
#   Llama-3.1-8B-Instruct (Meta; gated) -> Granite-3.3 / 3.1 8B Instruct (IBM; open) -> Qwen2.5-14B-Instruct -> Phi-3.5-mini (last resort)
GENERATOR_ID = GEN_INSTRUCT if FINAL_MODE == "instruct" else f"{GEN_BASE} + {ADAPTER_DIR.name} (fine-tuned)"
JUDGE_CANDIDATES = [(JUDGE_MODEL, 8), ("ibm-granite/granite-3.3-8b-instruct", 8), ("ibm-granite/granite-3.1-8b-instruct", 8),
                    ("Qwen/Qwen2.5-14B-Instruct", 14), ("Qwen/Qwen2.5-7B-Instruct", 7.6), ("microsoft/Phi-3.5-mini-instruct", 3.8)]
DONE_WITH = [GEN_BASE, GEN_INSTRUCT] + [n for n, _ in JUDGE_CANDIDATES]       # cached models this notebook no longer needs

JUDGE_USED, jtok, jmodel = None, None, None
for name, params_b in JUDGE_CANDIDATES:
    if name == GENERATOR_ID:
        continue                                            # never judge with the generator itself
    # bfloat16 when the GPU has room (2 bytes per parameter + headroom), otherwise 4-bit NF4
    bf16 = DEVICE == "cuda" and usable_gpu_gb() > 2.3 * params_b
    jkw = dict(dtype=COMPUTE_DTYPE) if bf16 else dict(quantization_config=BNB)
    try:
        check_resources(name, vram_gb=(2.3 if bf16 else 0.8) * params_b, finished_with=[m for m in DONE_WITH if m != name])
        jtok = AutoTokenizer.from_pretrained(name)
        jmodel = AutoModelForCausalLM.from_pretrained(name, device_map=DMAP, **jkw).eval()
        JUDGE_USED = f"{name} ({'bfloat16' if bf16 else '4-bit NF4'})"; break
    except (OSError, torch.cuda.OutOfMemoryError) as e:     # gated, not found, no disk or no GPU memory: try the next one
        print(f"skipped {name}: {str(e).splitlines()[0][:160]}")
        jmodel = None; free_gpu()
assert JUDGE_USED, "no judge model could be loaded"
print("Judge model:", JUDGE_USED, "| generator:", GENERATOR_ID)

FORMAT_REMINDER = ("\n\nYour reply must be exactly four lines, with no other text and no formatting:\n"
                   "CORRECT: CORRECT or PARTLY CORRECT or WRONG\nSUPPORTED: YES or NO\nREASON: one sentence\nUNSUPPORTED: a short quote or NONE")

@torch.inference_mode()
def judge(prompt):
    ids = jtok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt",
                                   return_dict=True).to(jmodel.device)
    out = jmodel.generate(**ids, max_new_tokens=200, do_sample=False, pad_token_id=jtok.pad_token_id or jtok.eos_token_id)
    return jtok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def parse(raw):
    clean = re.sub(r"[*_`#>]", "", raw)                    # tolerate markdown such as **CORRECT:** or bullet markers
    g = lambda k: (re.search(rf"^\s*-?\s*{k}\s*:\s*(.+)$", clean, re.M | re.I) or [None, ""])[1].strip()
    c = g("CORRECT").upper()
    correct = "PARTLY" if c.startswith("PARTLY") else "CORRECT" if c.startswith("CORRECT") else "WRONG" if c.startswith("WRONG") else "UNPARSED"
    sup = g("SUPPORTED").upper(); supported = "YES" if sup.startswith("YES") else "NO" if sup.startswith("NO") else "UNPARSED"
    return correct, supported, g("REASON"), g("UNSUPPORTED")

JUDGE_RAW, rows = {}, []
for q in QUERIES:
    prompt = judge_prompt(q["query"], q["expected"], TOP3[q["id"]], ANS[q["id"]])
    raw = judge(prompt); verdict = parse(raw)
    if "UNPARSED" in verdict[:2]:                           # one retry with an explicit format reminder (still greedy)
        raw = judge(prompt + FORMAT_REMINDER); verdict = parse(raw)
    JUDGE_RAW[q["id"]] = raw
    correct, supported, reason, unsup = verdict
    rows.append({"query": q["id"], "question": q["query"], "correct": correct, "supported": supported, "reason": reason,
                 "unsupported quote": unsup})
d2_df = pd.DataFrame(rows); d2_df.to_csv(OUT_DIR / "d2_judge_results.csv", index=False)
json.dump(JUDGE_RAW, open(OUT_DIR / "d2_judge_raw.json", "w"), indent=1)
pd.set_option("display.max_colwidth", 300)
display(d2_df)
if "UNPARSED" in set(d2_df.correct) | set(d2_df.supported):
    for k, v in JUDGE_RAW.items():
        print(f"--- {k} raw judge output:\n{v}")
    raise AssertionError("judge output not in the required format even after a reminder - raw outputs printed above")

summary = pd.DataFrame([("Correct", int((d2_df.correct == "CORRECT").sum())), ("Partly correct", int((d2_df.correct == "PARTLY").sum())),
                        ("Wrong", int((d2_df.correct == "WRONG").sum())), ("Supported by the chunks", int((d2_df.supported == "YES").sum()))],
                       columns=["Check", "Count out of 10"])
summary.to_csv(OUT_DIR / "d2_summary.csv", index=False); display(summary)
print("Judge model:", JUDGE_USED, "| generator:", GENERATOR_USED)
del jmodel; free_gpu()


freed 16.3 GB of disk: removed ibm-granite/granite-3.3-8b-instruct from the Hugging Face cache
freed 7.6 GB of disk: removed microsoft/Phi-3.5-mini-instruct from the Hugging Face cache
skipped meta-llama/Llama-3.1-8B-Instruct: You are trying to access a gated repo.


config.json:   0%|          | 0.00/790 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

... (5 outputs left out)

Judge model: ibm-granite/granite-3.3-8b-instruct (bfloat16) | generator: Qwen/Qwen2.5-7B-Instruct


,query,question,correct,supported,reason,unsupported quote
0,P1,"Under the RBI KYC Directions, who is the beneficial owner when the customer is a company?",CORRECT,YES,"The system answer accurately matches the expected answer, defining the beneficial owner as a natural person(s) with a controlling ownership interest or exercising control through other means. This information is directly sourced from chunk [1].",NONE
1,P2,"What minimum haircut applies to Level 2A assets in a bank's stock of high-quality liquid assets, and what is the cap on Level 2 assets?",CORRECT,YES,"The system answer accurately states the minimum 15% haircut for Level 2A assets and the 40% cap on Level 2 assets, aligning with the expected answer. Both facts are supported by the provided chunks.",NONE
2,P3,With how many life insurers may a Corporate Agent (Life) have arrangements?,CORRECT,YES,"The system answer correctly states that a Corporate Agent (Life) may have arrangements with a maximum of nine life insurers, aligning with the expected answer. This information is explicitly provided in chunk [1].",NONE
3,E1,What RSF factor is specified by the Basel Framework for other contingent funding obligations in Table 1?,WRONG,YES,"The system answer ""Not found in the documents"" is incorrect as the expected answer is present in the retrieved chunks. The chunks specify that national supervisors may set the required stable funding factors based on national circumstances for other contingent funding obligations, including unco...",
4,E2,What measures are required regarding cross-border currency transportation under the FATF cash couriers guidelines?,CORRECT,YES,"The system answer accurately reflects the expected answer, mentioning the need for declaration/disclosure systems, legal authority for competent authorities to stop or restrain suspicious items, and the aim to prevent terrorist financing and money laundering. All these points are supported by th...",NONE
5,E3,"What specific declarations regarding previous regulatory registrations must key individuals submit under the IRDAI (Registration of Corporate Agent) Regulations, 2015?",PARTLY,YES,"The system answer correctly identifies the declaration of Fit & Proper criteria as a requirement, but it only mentions this single aspect without including the crucial detail from the expected answer that these individuals must also declare any past registration or licensing with regulatory bodi...",
6,E4,How frequently are domestic structural liquidity statements and overseas branch reports required to be submitted to the Reserve Bank of India?,CORRECT,YES,"The system answer accurately reflects the expected answer, stating that domestic structural liquidity statements are required to be reported to the Reserve Bank of India fortnightly and overseas branch reports are submitted monthly. This information is supported by chunks [1], [2], and [3].",
7,E5,"How must a bank proceed when local laws and regulations abroad prohibit the implementation of the Reserve Bank of India (Commercial Banks – Know Your Customer) Directions, 2025?",PARTLY,YES,"The system answer correctly identifies that the bank should inform the Reserve Bank of India when local laws prohibit the implementation of certain directions. However, it mistakenly refers to the ""Reserve Bank of India (Commercial Banks – Asset Liability Management) Directions, 2025"" instead of...",NONE
8,E6,"How are group entities defined and classified under the Reserve Bank of India (Commercial Banks – Undertaking of Financial Services) Directions, 2025?",WRONG,NO,"The system answer states ""Not found in the documents,"" which contradicts the expected answer that provides specific details about group entities under the Reserve Bank of India (Commercial Banks – Undertaking of Financial Services) Directions, 2025. None of the provided chunks contain the inform...","The expected answer mentions Paragraph 4(9A) and its content, which is not supported by any of the retrieved chunks."
9,

,Check,Count out of 10
0,Correct,6
1,Partly correct,2
2,Wrong,2
3,Supported by the chunks,9


Judge model: ibm-granite/granite-3.3-8b-instruct (bfloat16) | generator: Fallback instruct model: Qwen/Qwen2.5-7B-Instruct


### D2.2 — Spot-check the judge (3 random answers)

In [30]:
import gc, sys, torch
for n in ["jmodel", "model", "qmodel", "base", "gen_model"]:
    globals().pop(n, None)
sys.last_traceback = sys.last_value = sys.last_type = None   # a failed cell's traceback can keep GPU tensors alive
gc.collect(); torch.cuda.empty_cache()
print(f"GPU in use: {torch.cuda.memory_allocated()/1e9:.2f} GB, free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

GPU in use: 0.23 GB, free: 84.08 GB


In [31]:
random.seed(SEED)
spot = random.sample([q["id"] for q in QUERIES], 3)
for qid in spot:
    q = next(x for x in QUERIES if x["id"] == qid)
    display(Markdown(f"---\n#### {qid}: {q['query']}\n**Expected:** {q['expected']}\n\n**Answer:** {ANS[qid]}\n\n**Judge:** {d2_df.set_index('query').loc[qid, 'correct']} / supported {d2_df.set_index('query').loc[qid, 'supported']} — {d2_df.set_index('query').loc[qid, 'reason']}"))
    for n, i in enumerate(TOP3[qid], 1):
        display(Markdown(f"> **[{n}]** ({FINAL_CHUNKS[i]['name']}, p.{FINAL_CHUNKS[i]['page']}) {FINAL_CHUNKS[i]['text']}"))

---
#### P2: What minimum haircut applies to Level 2A assets in a bank's stock of high-quality liquid assets, and what is the cap on Level 2 assets?
**Expected:** A minimum 15 percent haircut on the current market value of each Level 2A asset; Level 2 assets (2A and 2B), after haircuts, may comprise no more than 40 percent of the overall stock of HQLA.

**Answer:** The minimum haircut applicable to Level 2A assets in a bank's stock of high-quality liquid assets is 15%, and the cap on Level 2 assets is no more than 40% of the overall stock of HQLA [3].

**Judge:** CORRECT / supported YES — The system answer accurately states the minimum 15% haircut for Level 2A assets and the 40% cap on Level 2 assets, aligning with the expected answer. Both facts are supported by the provided chunks.

> **[1]** (BASEL_basel-framework-full, p.848) ) to hold additional Level 2A assets in the stock of HQLA. These additional Level 2A assets must be subject to a minimum haircut of 20%, ie 5% higher than the 15% haircut applicable to Level 2A assets that are included in the 40% cap. The higher haircut should cover any additional price and market liquidity risks arising from increased holdings of Level 2A assets beyond the 40% cap and provide a disincentive for banks to use this option based on yield considerations.

> **[2]** (BASEL_basel-framework-full, p.859) . ensure that they observe the higher requirements; the application of higher haircuts to additional Level 2A assets (see LCR31.17); 438 and 3. the existence of any restriction on the use of Level 2A assets (ie to what extent banks will 4. be allowed to hold such assets as a percentage of their liquid asset stock). 1 Footnote 438 Under LCR31.17, a minimum higher haircut of 20% must be applied to additional Level 2A assets

> **[3]** (RBI_163MD, p.55) haircuts, shall comprise no more than 40 per cent of the overall stock of HQLA. The portfolio of Level 2 assets held by a bank shall be well diversified in terms of type of assets, type of issuers, and specific counterparty or issuer. Level 2 assets shall be subject to fundamental characteristics and market related characteristics of HQLA. 143. Level 2A Assets (1) A minimum 15 per cent haircut shall be applied to the current market value of each Level 2A asset held

---
#### P1: Under the RBI KYC Directions, who is the beneficial owner when the customer is a company?
**Expected:** The natural person(s) who, alone or together, or through one or more juridical persons, has a controlling ownership interest (ownership of / entitlement to more than 10 percent of the shares, capital or profits of the company) or who exercises control through other means (e.g. the right to appoint the majority of directors or control management or policy decisions).

**Answer:** The beneficial owner is the natural person(s) who, whether acting alone or together, or through one or more juridical persons, has/have a controlling ownership interest or who exercises control through other means [1].

**Judge:** CORRECT / supported YES — The system answer accurately matches the expected answer, defining the beneficial owner as a natural person(s) with a controlling ownership interest or exercising control through other means. This information is directly sourced from chunk [1].

> **[1]** (RBI_169MD, p.5) means the process as defined under sub-section (c) of section 2 of the Aadhaar (Targeted Delivery of Financial and Other Subsidies, Benefits and Services) Act, 2016. (iv) Beneficial Owner (BO) (a) Where the customer is a company, the beneficial owner is the natural person(s), who, whether acting alone or together, or through one or more juridical persons, has / have a controlling ownership interest or who exercises control through other means. Explanation

... (2 outputs left out)

---
#### E2: What measures are required regarding cross-border currency transportation under the FATF cash couriers guidelines?
**Expected:** Countries must establish declaration or disclosure systems to detect physical cross-border transportations of currency and bearer negotiable instruments. Competent authorities need legal authority to stop or restrain suspicious or falsely declared items, and effective, proportionate, and dissuasive sanctions must apply to false declarations alongside confiscation measures for illicit funds.

**Answer:** Countries are required to have measures in place to detect the physical cross-border transportation of currency and bearer negotiable instruments, which can include a declaration or disclosure system. Additionally, competent authorities must have the legal authority to stop or restrain such items if they are suspected to be related to certain criminal activities [1]. These measures aim to prevent the financing of terrorist activities and money laundering through such transportation [2].

**Judge:** CORRECT / supported YES — The system answer accurately reflects the expected answer, mentioning the need for declaration/disclosure systems, legal authority for competent authorities to stop or restrain suspicious items, and the aim to prevent terrorist financing and money laundering. All these points are supported by the provided chunks.

> **[1]** (FATF_fatf-recommendations-2012, p.26) Cash couriers * Countries should have measures in place to detect the physical cross-border transportation of currency and bearer negotiable instruments, including through a declaration system and/or disclosure system. Countries should ensure that their competent authorities have the legal authority to stop or restrain currency or bearer negotiable instruments that are suspected to be related to terrorist financing, money laundering or predicate offences, or that are falsely declared or disclosed. Countries should ensure that effective, proportionate and dissuasive

> **[2]** (FATF_fatf-recommendations-2012, p.115) 32 (CASH COURIERS) A. OBJECTIVES 1. Recommendation 32 was developed with the objective of ensuring that terrorists and other criminals cannot finance their activities or launder the proceeds of their crimes through the physical cross-border transportation of currency and bearer negotiable instruments. Specifically, it aims to ensure that countries have measures to: (a) detect the physical cross-border transportation of currency and bearer negotiable instruments; (b) stop or restrain currency and bearer negotiable instruments

> **[3]** (FATF_fatf-recommendations-2012, p.117) . This includes failing to make a disclosure as required. Glossary of specific terms used in this Recommendation Physical cross-border transportation refers to any in-bound or out-bound physical transportation of currency or BNIs from one country to another country. The term includes the following modes of transportation: (1) physical transportation by a natural person, or in that person’s accompanying luggage or vehicle; (2) shipment of currency or BNIs through containerised cargo or (3) the mailing of

**D2.2: our verdict on the three random answers (P2, P1, E2).**

- **SUPPORTED:** we agree with the judge on all 3; every statement in the three answers can be found in their chunks.
- **CORRECT:** we agree on **1 of 3** (P2: 15% haircut and 40% cap, both stated, so CORRECT).
- **Disagreements, P1 and E2:** the judge labelled both CORRECT, but its own definition requires *every* key fact.
  - **P1:** the answer omits the "more than 10 percent" ownership threshold. That threshold is in the expected answer, but it was not in the chunks: chunk [1] ends at "Explanation", just before the threshold.
  - **E2:** the answer omits the sanctions for false declarations and the confiscation measures.

  Both should be **PARTLY CORRECT**. The judge is lenient when an answer states the main rule but leaves out a qualifying detail.
- **Outside the sample, E6:** the judge marked the "Not found in the documents" reply SUPPORTED: NO. Our prompt defines that reply as SUPPORTED: YES, because it makes no claim.

### D2.3 — Find the cause of one failed answer

In [32]:
failed = d2_df[(d2_df.correct == "WRONG") | (d2_df.supported == "NO")]
pick = (failed[failed.correct == "WRONG"] if (failed.correct == "WRONG").any() else failed)
if pick.empty:
    pick = d2_df[d2_df.correct == "PARTLY"]
FAIL_ID = pick.iloc[0]["query"] if not pick.empty else QUERIES[0]["id"]
q = next(x for x in QUERIES if x["id"] == FAIL_ID)
span_in_chunks = any(is_hit(FINAL_CHUNKS[i], SPANS[FAIL_ID]) for i in TOP3[FAIL_ID])
print(f"Query {FAIL_ID}: {q['query']}\nExpected: {q['expected']}\nAnswer: {ANS[FAIL_ID]}\nJudge: {d2_df.set_index('query').loc[FAIL_ID].to_dict()}")
for n, i in enumerate(TOP3[FAIL_ID], 1):
    print(f"\n[{n}] chunk #{i} ({FINAL_CHUNKS[i]['name']}, p.{FINAL_CHUNKS[i]['page']}) contains answer span: {is_hit(FINAL_CHUNKS[i], SPANS[FAIL_ID])}\n{FINAL_CHUNKS[i]['text']}")
print("\nAnswer span present in the 3 chunks:", span_in_chunks,
      "->", "GENERATION problem (information was supplied but not used correctly)" if span_in_chunks else "RETRIEVAL problem (the needed information never reached the model)")
print("Position of the span in Part B / C rankings: first-hit rank before rerank =", first_hit(FAIL_ID, C[[c['id'] for c in C].index(FAIL_ID)]['before']),
      "| after =", first_hit(FAIL_ID, C[[c['id'] for c in C].index(FAIL_ID)]['after']))

Query E1: What RSF factor is specified by the Basel Framework for other contingent funding obligations in Table 1?
Expected: Table 1 of the Basel Framework specifies that national supervisors may set the required stable funding factors based on national circumstances for other contingent funding obligations. These include unconditionally revocable facilities, trade finance obligations, guarantees, and certain non-contractual obligations.
Answer: Not found in the documents.
Judge: {'question': 'What RSF factor is specified by the Basel Framework for other contingent funding obligations in Table 1?', 'correct': 'WRONG', 'supported': 'YES', 'reason': 'The system answer "Not found in the documents" is incorrect as the expected answer is present in the retrieved chunks. The chunks specify that national supervisors may set the required stable funding factors based on national circumstances for other contingent funding obligations, including unconditionally revocable facilities, trade finance

**D2.3: cause.**

The failure of query **E1** is a **generation** problem: the needed information was in the 3 chunks.
- **Chunk [1] holds the answer span.** Reranking had moved it from 3rd to 1st. It contains the Table 1 row "National supervisors may specify the RSF factors based on their national circumstances. Other contingent funding obligations …".
- **The model still replied "Not found in the documents".** In the extracted text, the table is flattened into one line, and each row's RSF factor comes before its category. The model therefore did not link "national supervisors may specify" to "other contingent funding obligations".
- **It was also looking for a number.** The question asks "what RSF factor", and the correct answer is a rule (supervisors set it), not a figure.

### D2.4 — Suggested fix

We would change the generation prompt by adding two rules:
- "Context may include tables flattened into text, where a row's value can appear before its label; read each row as label + value."
- "If the chunks answer the question only partly, or the answer is a rule rather than a number, give what they say and state what is missing; reply 'Not found in the documents' only if no chunk is relevant."

This targets the cause found above. Retrieval already delivered the right chunk at rank 1, so the failure lies between the chunk and the answer. The same rule would also stop the model discarding partial evidence. A longer-term fix is table-aware extraction in the corpus pipeline (one "category: RSF factor" line per row), so that tables reach the model already linearised.

## Submission checklist and export

In [ ]:
need = ["a1_index_report.csv", "a2_chunk_size.csv", "b_search_comparison.csv", "c_rerank_table.csv", "c_before_after.csv",
        "d1_answers.csv", "d2_judge_results.csv", "d2_summary.csv"]
display(pd.DataFrame([{"saved output": n, "present": (OUT_DIR / n).exists()} for n in need]))
print("Outputs folder:", OUT_DIR)

In [ ]:
# HTML export of this notebook (run after saving)
!jupyter nbconvert --to html "Assignment_2B_RAG_Pipeline.ipynb"